<a href="https://colab.research.google.com/github/lukalklikadze/walmart-sales-forecasting/blob/main/notebooks/prophet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ══ Bootstrap ══
!pip -q install mlflow kaggle prophet

!git clone -q https://github.com/lukalklikadze/walmart-sales-forecasting.git /content/repo \
    2>/dev/null || (cd /content/repo && git pull -q)
import sys; sys.path.append("/content/repo")

from src.config import setup_env, DATA_DIR, KAGGLE_COMP
print("MLflow →", setup_env())

import os, glob, zipfile, subprocess
os.makedirs(DATA_DIR, exist_ok=True)
subprocess.run(["kaggle","competitions","download","-c",KAGGLE_COMP,"-p",DATA_DIR,"--force"], check=True)
with zipfile.ZipFile(os.path.join(DATA_DIR, KAGGLE_COMP + ".zip")) as z: z.extractall(DATA_DIR)
for inner in glob.glob(os.path.join(DATA_DIR, "*.csv.zip")):
    with zipfile.ZipFile(inner) as z: z.extractall(DATA_DIR)

from src.data import load_raw
train, test, stores, features = load_raw()
print("train:", train.shape, "| test:", test.shape)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/

# Prophet

Prophet models one series at a time (trend + yearly seasonality + holiday effects) and can't
share anything between series. So we fit a separate Prophet for every (Store, Dept) - 3331
models - and wrap them all in one sklearn-style class, so the result is still a single
pipeline that runs on the raw test frame.

The part that makes Prophet interesting for us is its holiday handling, because it fits what
we saw in the EDA: we register the 4 kaggle holidays as separate named events, plus the week
before christmas - the week where the actual christmas shopping happens, which the IsHoliday
flag misses. This way every holiday gets its own effect instead of one averaged IsHoliday
coefficient. The second run drops all holiday events, so the difference between the two runs
tells us what the holiday modelling is actually worth (holiday weeks weigh 5x in the metric).

Keeping 3331 fits manageable: they're independent, so joblib runs them in parallel;
uncertainty sampling is off (expensive and we don't use it); weekly/daily seasonality is off
since the data is already weekly; and series with under 60 weeks of history just predict
their mean instead of getting an unstable fit. To smoke-test the plumbing first you can
temporarily fit on a slice like train[train.Store <= 3].

In [ ]:
import time, logging
import numpy as np, pandas as pd
import mlflow, mlflow.sklearn
from sklearn.pipeline import Pipeline

from src.train_val_split import time_based_split
from src.metrics import wmae

# prophet/cmdstanpy are very chatty otherwise
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)

# last 39 weeks of train as validation - same length as the real test horizon
VAL_WEEKS = 39
tr, va = time_based_split(train, val_weeks=VAL_WEEKS)
Xtr, ytr = tr.drop(columns=["Weekly_Sales"]), tr["Weekly_Sales"]
Xva, yva = va.drop(columns=["Weekly_Sales"]), va["Weekly_Sales"]
X_full, y_full = train.drop(columns=["Weekly_Sales"]), train["Weekly_Sales"]
print(f"train fold {tr.Date.min().date()}..{tr.Date.max().date()} ({len(tr):,} rows)")
print(f"val fold   {va.Date.min().date()}..{va.Date.max().date()} "
      f"({len(va):,} rows, {int(va.IsHoliday.sum()):,} holiday rows)")

train fold 2010-02-05..2012-01-27 (305,982 rows)
val fold   2012-02-03..2012-10-26 (115,588 rows, 5,967 holiday rows)


In [ ]:
from joblib import Parallel, delayed
from prophet import Prophet
from sklearn.base import BaseEstimator, RegressorMixin


def kaggle_holidays():
    '''The 4 flagged kaggle holidays + the unflagged pre-christmas week, as
    named prophet events (train and test occurrences).'''
    def block(name, dates):
        return pd.DataFrame({"holiday": name, "ds": pd.to_datetime(dates)})
    return pd.concat([
        block("super_bowl",   ["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"]),
        block("labor_day",    ["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"]),
        block("thanksgiving", ["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"]),
        block("christmas",    ["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"]),
        block("week_before_christmas",
              ["2010-12-24", "2011-12-23", "2012-12-21", "2013-12-20"]),
    ], ignore_index=True)


def _fit_one(key, sub, holidays):
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                daily_seasonality=False, holidays=holidays,
                uncertainty_samples=0)
    m.fit(sub.rename(columns={"Date": "ds"})[["ds", "y"]])
    return key, m


class ProphetPerSeries(BaseEstimator, RegressorMixin):
    '''One Prophet per (Store, Dept), wrapped so it lives in a Pipeline.

    fit trains the series with at least min_history weeks in parallel and
    keeps per-series means (plus a global median) as fallback for the rest.
    predict extrapolates each series' trend/seasonality/holidays to the
    requested dates; fallback series just get their mean.
    '''

    def __init__(self, use_holidays=True, min_history=60, n_jobs=-1):
        self.use_holidays = use_holidays
        self.min_history = min_history
        self.n_jobs = n_jobs

    def fit(self, X, y):
        df = X[["Store", "Dept", "Date"]].copy()
        df["y"] = np.asarray(y, dtype=float)
        df = df.sort_values(["Store", "Dept", "Date"])
        hol = kaggle_holidays() if self.use_holidays else None
        groups = list(df.groupby(["Store", "Dept"]))
        eligible = [(k, g) for k, g in groups if len(g) >= self.min_history]
        print(f"fitting {len(eligible)} series in parallel "
              f"(skipping {len(groups) - len(eligible)} with <{self.min_history} weeks)")
        fitted = Parallel(n_jobs=self.n_jobs, verbose=1)(
            delayed(_fit_one)(k, g, hol) for k, g in eligible)
        self.models_ = dict(fitted)
        self.series_mean_ = df.groupby(["Store", "Dept"])["y"].mean()
        self.global_median_ = float(df["y"].median())
        return self

    def predict(self, X):
        out = np.full(len(X), np.nan)
        pos = pd.Series(np.arange(len(X)), index=X.index)
        for key, g in X.groupby(["Store", "Dept"]):
            i = pos.loc[g.index].to_numpy()
            m = self.models_.get(key)
            if m is not None:
                out[i] = m.predict(pd.DataFrame({"ds": g["Date"]}))["yhat"].to_numpy()
            elif key in self.series_mean_.index:
                out[i] = self.series_mean_.loc[key]
            else:
                out[i] = self.global_median_
        return out

In [ ]:
mlflow.set_experiment("Prophet_Training")
RESULTS = {}

def run_prophet(run_name, **kw):
    with mlflow.start_run(run_name=run_name):
        pipe = Pipeline([("model", ProphetPerSeries(**kw))])
        t0 = time.time(); pipe.fit(Xtr, ytr)
        score = wmae(yva, pipe.predict(Xva), Xva["IsHoliday"])
        mlflow.log_params(pipe.named_steps["model"].get_params())
        mlflow.log_metrics({"val_wmae": score, "fit_seconds": time.time() - t0})
        RESULTS[run_name] = (score, kw)
        print(f"{run_name:24s} val WMAE {score:9.1f}")
        return score

# run 1: with the per-holiday events (incl. week_before_christmas)
run_prophet("prophet_holidays", use_holidays=True)

2026/07/09 10:00:17 INFO mlflow.tracking.fluent: Experiment with name 'Prophet_Training' does not exist. Creating a new experiment.


fitting 2925 series in parallel (skipping 381 with <60 weeks)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:   15.1s
[Parallel(n_jobs=-1)]: Done 282 tasks      | elapsed:   29.7s
[Parallel(n_jobs=-1)]: Done 782 tasks      | elapsed:   59.9s
[Parallel(n_jobs=-1)]: Done 1482 tasks      | elapsed:  1.7min
[Parallel(n_jobs=-1)]: Done 2382 tasks      | elapsed:  2.7min
[Parallel(n_jobs=-1)]: Done 2922 out of 2925 | elapsed:  3.3min remaining:    0.2s
[Parallel(n_jobs=-1)]: Done 2925 out of 2925 | elapsed:  3.3min finished


prophet_holidays         val WMAE    1764.6
🏃 View run prophet_holidays at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/4/runs/603b82165dc84a2ab325a21672fdcdce
🧪 View experiment at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/4


1764.5656320496494

In [ ]:
# run 2: same thing without holidays - the gap between the two runs is what
# explicit holiday modelling is worth under the 5x weighting
run_prophet("prophet_no_holidays", use_holidays=False)

fitting 2925 series in parallel (skipping 381 with <60 weeks)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done 164 tasks      | elapsed:    8.6s
[Parallel(n_jobs=-1)]: Done 764 tasks      | elapsed:   37.9s
[Parallel(n_jobs=-1)]: Done 1764 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done 2925 out of 2925 | elapsed:  2.6min finished


prophet_no_holidays      val WMAE    1742.7
🏃 View run prophet_no_holidays at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/4/runs/aa5353f57c3448a8ba736dd93a4577b6
🧪 View experiment at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/4


1742.7246521502716

In [ ]:
# refit the better config on the whole train set and log the final pipeline
best_name = min(RESULTS, key=lambda k: RESULTS[k][0])
score, best_kw = RESULTS[best_name]
print("refitting on full train:", best_name, best_kw)

final_pipe = Pipeline([("model", ProphetPerSeries(**best_kw))])
with mlflow.start_run(run_name=f"final_full_train__{best_name}"):
    t0 = time.time(); final_pipe.fit(X_full, y_full)
    preds = final_pipe.predict(test)
    assert np.isfinite(preds).all()
    mlflow.log_params({"chosen_run": best_name, "refit_on": "full_train", **best_kw})
    mlflow.log_metrics({"val_wmae_of_chosen_cfg": score,
                        "fit_seconds": time.time() - t0})
    try:
        mlflow.sklearn.log_model(final_pipe, "model",
                             serialization_format="cloudpickle")
    except Exception as e:   # 3331 pickled prophets can get big
        print("log_model failed, logging joblib artifact instead:", e)
        import joblib
        joblib.dump(final_pipe, "prophet_pipeline.joblib")
        mlflow.log_artifact("prophet_pipeline.joblib", "model")
print("sample test preds:", preds[:3].round(1))

refitting on full train: prophet_no_holidays {'use_holidays': False}
fitting 2971 series in parallel (skipping 360 with <60 weeks)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  88 tasks      | elapsed:    3.5s
[Parallel(n_jobs=-1)]: Done 388 tasks      | elapsed:   17.3s
[Parallel(n_jobs=-1)]: Done 888 tasks      | elapsed:   42.0s
[Parallel(n_jobs=-1)]: Done 1588 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done 2488 tasks      | elapsed:  2.0min
[Parallel(n_jobs=-1)]: Done 2971 out of 2971 | elapsed:  2.4min finished
2026/07/09 10:11:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/09 10:11:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run final_full_train__prophet_no_holidays at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/4/runs/7c915f5845124df3868bceffd9ffa7d7
🧪 View experiment at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/4
sample test preds: [33084.9 27507.  19564.1]


Prophet is our "interpretable classical" option - every series gets its own readable
trend/seasonality/holiday decomposition, but nothing is shared between series, and ~2.7
yearly cycles of history per series is not much. The holidays vs no_holidays comparison puts
an actual number on the per-holiday idea from the EDA.